# Building an Agent That Can Search the Web

In this notebook, we will build an AI agent that has the power to **search the internet** in real time. Unlike a basic chatbot that only knows what it was trained on, this agent can look up current information to answer your questions.

We will use Microsoft Foundry's `WebSearchPreviewTool` to give our agent live web access.

## Step 1: Install Packages

Run the cell below to install all the Python packages we need.

In [ ]:
# Install the required libraries for working with Foundry agents and web search
%pip install azure-ai-projects==2.0.0b3 openai==1.109.1 python-dotenv azure-identity

## Step 2: Load Settings and Import Libraries

We load our project settings from a `.env` file and import all the classes we need. Notice we are also importing `WebSearchPreviewTool` — this is the tool that gives our agent internet access.

In [ ]:
# Standard library for accessing environment variables
import os

# Reads key-value pairs from a .env file into the environment
from dotenv import load_dotenv

# Handles Azure login credentials automatically
from azure.identity import DefaultAzureCredential

# The main client for connecting to a Foundry project
from azure.ai.projects import AIProjectClient

# PromptAgentDefinition: defines what our agent looks like and how it behaves
# WebSearchPreviewTool: a built-in tool that lets the agent search the web
from azure.ai.projects.models import PromptAgentDefinition, WebSearchPreviewTool

# Load the .env file so we can read our settings
load_dotenv()

# The URL of our Foundry project
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")

# The name of the deployed AI model we want the agent to use
my_model = os.getenv("MODEL_DEPLOYMENT_NAME")

## Step 3: Connect to Microsoft Foundry

We create a Foundry client using our endpoint and Azure credentials. This client is our gateway to creating agents and conversations.

In [ ]:
# Open a connection to our Foundry project in the cloud
foundry_client = AIProjectClient(
    endpoint=my_endpoint,
    credential=DefaultAzureCredential(),
)

## Step 4: Get the OpenAI Chat Client

The chat client is what we use to send messages and receive responses. Foundry provides an OpenAI-compatible client, so the API feels familiar if you have used OpenAI before.

In [ ]:
# Get an OpenAI-compatible client for sending messages to agents
chat_client = foundry_client.get_openai_client()

## Step 5: Create a Web-Enabled Agent

Here we create an agent that has the `WebSearchPreviewTool` attached. This tool allows the agent to perform live web searches when answering questions, so it can find up-to-date information instead of relying only on its training data.

In [ ]:
# Choose a unique name for this agent
search_agent_name = "live-search-assistant"

# Create the web-enabled search agent
# - agent_name: a unique identifier for this agent
# - model: the AI model powering the agent
# - instructions: tells the agent how to behave
# - tools: a list of tools the agent can use (in this case, web search)
search_agent = foundry_client.agents.create_version(
    agent_name=search_agent_name,
    definition=PromptAgentDefinition(
        model=my_model,
        instructions="You are a research assistant that searches the web to find current, accurate answers to user questions.",
        tools=[
            WebSearchPreviewTool()  # This gives the agent the ability to search the internet
        ]
    )
)

## Step 6: Open a Chat Session

We create a new conversation so the agent can track context across multiple messages.

In [ ]:
# Start a fresh conversation — this stores the message history
chat_session = chat_client.conversations.create()
print(f"Created conversation with id: {chat_session.id}")

## Step 7: Ask a Question and Get Live Results

Now let us ask our agent a question that requires up-to-date information. The agent will search the web and return a summarized answer.

In [ ]:
# Define the question we want to ask — this is something that needs live web data
question = "What are the top trending technologies in 2026?"

In [ ]:
# Send our question to the web-search agent and wait for the response
# - conversation: keeps this message in context with the chat session
# - extra_body: tells Foundry to route this to our search agent
# - input: the question we want answered
response = chat_client.responses.create(
    conversation=chat_session.id,
    extra_body={
        "agent": {
            "name": search_agent_name,
            "type": "agent_reference"
        }
    },
    input=question
)

# Print the agent's answer (which includes information from live web results)
print(f"Agent response: {response.output_text}")